# Demo simple - Outliers en dos subpartidas

Notebook simple para presentar la tesis: se carga la data desde SQL Server, se filtra solo `806100000` y `810400000`, se generan folds dentro de cada subpartida y se comparan modelos de detección de outliers sobre el valor unitario.


In [1]:
# =============================================================================
# 1. CONFIGURACION SIMPLE
# =============================================================================

import warnings
import time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool
from sklearn.model_selection import KFold
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

SEED = 42
SERVIDOR = r"DESKTOP-OGU19A7\SQLEXPRESS,56878"
BASE_DATOS = "DB_GEE_DW_ADUANAS"
ESQUEMA = "SC_ADUANA"
PROCEDIMIENTO = "SP_VALORES_UNITARIOS"
DRIVER = "ODBC+Driver+17+for+SQL+Server"

FEC_INI = "2024-01-01"
FEC_FIN = "2024-12-31"

MIN_REGISTROS = 36
N_FOLDS = 5
CONTAMINACION = 0.05

print("Configuracion cargada")
print(f"Servidor  : {SERVIDOR}")
print(f"BD        : {BASE_DATOS}")
print(f"SP        : {ESQUEMA}.{PROCEDIMIENTO}")
print(f"Periodo   : {FEC_INI} a {FEC_FIN}")
print("Subpartidas: 806100000 y 810400000")

Configuracion cargada
Servidor  : DESKTOP-OGU19A7\SQLEXPRESS,56878
BD        : DB_GEE_DW_ADUANAS
SP        : SC_ADUANA.SP_VALORES_UNITARIOS
Periodo   : 2024-01-01 a 2024-12-31
Subpartidas: 806100000 y 810400000


In [2]:
# =============================================================================
# 2. CARGA DIRECTA DESDE SQL SERVER
# =============================================================================

inicio = time.perf_counter()

url = (
    f"mssql+pyodbc://{SERVIDOR}/{BASE_DATOS}"
    f"?driver={DRIVER}&Trusted_Connection=yes"
)

motor = create_engine(url, echo=False, poolclass=NullPool)

try:
    sentencia = text(
        f"EXEC [{BASE_DATOS}].[{ESQUEMA}].[{PROCEDIMIENTO}] "
        f"@ACCION='EDA_BASE', "
        f"@FEC_INI='{FEC_INI}', "
        f"@FEC_FIN='{FEC_FIN}'"
    )

    with motor.connect() as conn:
        resultado = conn.execute(sentencia)
        df = pd.DataFrame.from_records(resultado.fetchall(), columns=list(resultado.keys()))
finally:
    motor.dispose()

if df.empty:
    raise ValueError("El SP retorno 0 registros. Revisa la ingesta o el rango de fechas.")

# Tipos simples segun lo que devuelve EDA_BASE.
df["NUM_SPN_R"] = df["NUM_SPN_R"].astype(str)
df["ANIO_C"] = pd.to_numeric(df["ANIO_C"], errors="coerce").astype("Int64")
df["MTO_VALOR_UNTARIO_V"] = pd.to_numeric(df["MTO_VALOR_UNTARIO_V"], errors="coerce")
df["FOB_DOLAR"] = pd.to_numeric(df["FOB_DOLAR"], errors="coerce")
df["PESO_NETO"] = pd.to_numeric(df["PESO_NETO"], errors="coerce")

# Limpieza minima.
df = df[
    df["NUM_SPN_R"].notna()
    & df["ANIO_C"].notna()
    & df["MTO_VALOR_UNTARIO_V"].notna()
    & df["FOB_DOLAR"].notna()
    & df["PESO_NETO"].notna()
    & (df["MTO_VALOR_UNTARIO_V"] > 0)
    & (df["FOB_DOLAR"] > 0)
    & (df["PESO_NETO"] > 0)
].copy()

df["LOG_VALOR_UNITARIO"] = np.log1p(df["MTO_VALOR_UNTARIO_V"])
df["ID_REGISTRO"] = np.arange(1, len(df) + 1)

print(f"Registros : {len(df):,}")
print(f"Partidas  : {df['NUM_SPN_R'].nunique():,}")
print(f"Anios     : {sorted(df['ANIO_C'].dropna().unique().tolist())}")
print(f"Columnas  : {list(df.columns)}")
print(f"Tiempo    : {time.perf_counter() - inicio:.2f} segundos")

df.head()

Registros : 390,257
Partidas  : 786
Anios     : [2024]
Columnas  : ['ANIO_C', 'NUM_SPN_R', 'MTO_VALOR_UNTARIO_V', 'FOB_DOLAR', 'PESO_NETO', 'SECTOR', 'TIPO_PRODUCTO', 'ADUANA', 'LOG_VALOR_UNITARIO', 'ID_REGISTRO']
Tiempo    : 6.73 segundos


,ANIO_C,NUM_SPN_R,MTO_VALOR_UNTARIO_V,FOB_DOLAR,PESO_NETO,SECTOR,TIPO_PRODUCTO,ADUANA,LOG_VALOR_UNITARIO,ID_REGISTRO
0,2024,"2005700000-ACEITUNAS PREPARADAS O CONSERVADAS,...",3.649023,3773.09,1034.0,500,02 - PRODUCTOS NO TRADICIONALES,MARITIMA DEL CALLAO,1.536657,1
1,2024,"804200000-HIGOS, FRESCOS O SECOS",10.829752,10483.20,968.0,500,02 - PRODUCTOS NO TRADICIONALES,AEREA Y POSTAL EX - IAAC,2.470618,2
2,2024,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",2.742683,54853.66,20000.0,500,02 - PRODUCTOS NO TRADICIONALES,MARITIMA DEL CALLAO,1.319803,3
3,2024,"1008509000-QUINUA, EXCEPTO PARA LA SIEMBRA",1.179902,23598.04,20000.0,500,02 - PRODUCTOS NO TRADICIONALES,MARITIMA DEL CALLAO,0.779280,4
4,2024,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",11.333333,4080.00,360.0,500,02 - PRODUCTOS NO TRADICIONALES,AEREA Y POSTAL EX - IAAC,2.512306,5


In [3]:
# =============================================================================
# 3. SELECCION SIMPLE DE SUBPARTIDAS PARA LA DEMO
# =============================================================================

# El SP devuelve NUM_SPN_R como texto: num_partida + '-' + partida.
# Por eso se filtra por el inicio del texto.
df_demo = df[
    df["NUM_SPN_R"].astype(str).str.startswith(("806100000", "810400000"))
].copy()

conteo_demo = (
    df_demo.groupby("NUM_SPN_R")
    .size()
    .reset_index(name="REGISTROS")
    .sort_values("NUM_SPN_R")
)

if df_demo.empty:
    raise ValueError("No hay datos para las subpartidas 806100000 y 810400000.")

print("Subpartidas seleccionadas: 806100000 y 810400000")
print(f"Registros demo: {len(df_demo):,}")

conteo_demo


Subpartidas seleccionadas: 806100000 y 810400000
Registros demo: 53,106


,NUM_SPN_R,REGISTROS
0,806100000-UVAS FRESCAS,25733
1,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",27373


In [4]:
# =============================================================================
# 4. FOLDS DENTRO DE CADA SUBPARTIDA
# =============================================================================

folds = []

for subpartida, df_sub in df_demo.groupby("NUM_SPN_R"):
    df_sub = df_sub.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_splits = min(N_FOLDS, len(df_sub))

    if n_splits < 2:
        continue

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for fold_id, (idx_train, idx_test) in enumerate(kfold.split(df_sub), start=1):
        folds.append({
            "subpartida": subpartida,
            "fold": fold_id,
            "train": df_sub.iloc[idx_train].copy(),
            "test": df_sub.iloc[idx_test].copy(),
        })

if not folds:
    raise ValueError("No se crearon folds. Revisa que las subpartidas 806100000 y 810400000 tengan registros suficientes.")

resumen_folds = pd.DataFrame([
    {
        "subpartida": f["subpartida"],
        "fold": f["fold"],
        "n_train": len(f["train"]),
        "n_test": len(f["test"]),
    }
    for f in folds
])

resumen_folds.head(10)

,subpartida,fold,n_train,n_test
0,806100000-UVAS FRESCAS,1,20586,5147
1,806100000-UVAS FRESCAS,2,20586,5147
2,806100000-UVAS FRESCAS,3,20586,5147
3,806100000-UVAS FRESCAS,4,20587,5146
4,806100000-UVAS FRESCAS,5,20587,5146
5,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",1,21898,5475
6,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",2,21898,5475
7,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",3,21898,5475
8,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",4,21899,5474
9,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",5,21899,5474


In [5]:
# =============================================================================
# 5. VALIDACION SIMPLE CON OUTLIERS SINTETICOS
# =============================================================================

resultados = []

for item in folds:
    subpartida = item["subpartida"]
    fold_id = item["fold"]
    train = item["train"].copy()
    test = item["test"].copy()

    n_sint = max(2, int(len(test) * CONTAMINACION))
    sint = test.sample(n=min(n_sint, len(test)), replace=True, random_state=SEED).copy()

    mediana_vu = train["MTO_VALOR_UNTARIO_V"].median()
    mitad = len(sint) // 2
    sint["MTO_VALOR_UNTARIO_V"] = mediana_vu
    sint.iloc[:mitad, sint.columns.get_loc("MTO_VALOR_UNTARIO_V")] = mediana_vu * 0.08
    sint.iloc[mitad:, sint.columns.get_loc("MTO_VALOR_UNTARIO_V")] = mediana_vu * 6.00
    sint["LOG_VALOR_UNITARIO"] = np.log1p(sint["MTO_VALOR_UNTARIO_V"])

    test["y_true"] = 0
    sint["y_true"] = 1
    test_eval = pd.concat([test, sint], ignore_index=True)

    # Modelo 1: IQR por subpartida.
    q1 = train["MTO_VALOR_UNTARIO_V"].quantile(0.25)
    q3 = train["MTO_VALOR_UNTARIO_V"].quantile(0.75)
    iqr = q3 - q1
    li = q1 - 1.5 * iqr
    ls = q3 + 1.5 * iqr
    pred_iqr = ((test_eval["MTO_VALOR_UNTARIO_V"] < li) | (test_eval["MTO_VALOR_UNTARIO_V"] > ls)).astype(int)

    resultados.append({
        "subpartida": subpartida,
        "fold": fold_id,
        "modelo": "IQR",
        "precision": precision_score(test_eval["y_true"], pred_iqr, zero_division=0),
        "recall": recall_score(test_eval["y_true"], pred_iqr, zero_division=0),
        "f1": f1_score(test_eval["y_true"], pred_iqr, zero_division=0),
    })

    # Modelo 2: Isolation Forest.
    iso = IsolationForest(contamination=CONTAMINACION, random_state=SEED)
    iso.fit(train[["LOG_VALOR_UNITARIO"]])
    pred_iso = (iso.predict(test_eval[["LOG_VALOR_UNITARIO"]]) == -1).astype(int)

    resultados.append({
        "subpartida": subpartida,
        "fold": fold_id,
        "modelo": "IsolationForest",
        "precision": precision_score(test_eval["y_true"], pred_iso, zero_division=0),
        "recall": recall_score(test_eval["y_true"], pred_iso, zero_division=0),
        "f1": f1_score(test_eval["y_true"], pred_iso, zero_division=0),
    })

    # Modelo 3: LOF novelty.
    n_neighbors = min(20, max(2, len(train) - 1))
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=CONTAMINACION, novelty=True)
    lof.fit(train[["LOG_VALOR_UNITARIO"]])
    pred_lof = (lof.predict(test_eval[["LOG_VALOR_UNITARIO"]]) == -1).astype(int)

    resultados.append({
        "subpartida": subpartida,
        "fold": fold_id,
        "modelo": "LOF",
        "precision": precision_score(test_eval["y_true"], pred_lof, zero_division=0),
        "recall": recall_score(test_eval["y_true"], pred_lof, zero_division=0),
        "f1": f1_score(test_eval["y_true"], pred_lof, zero_division=0),
    })

tabla_resultados = pd.DataFrame(resultados)

tabla_resumen = (
    tabla_resultados
    .groupby(["subpartida", "modelo"], as_index=False)
    .agg(
        precision_prom=("precision", "mean"),
        recall_prom=("recall", "mean"),
        f1_prom=("f1", "mean"),
    )
    .sort_values(["subpartida", "f1_prom"], ascending=[True, False])
)

tabla_resumen

,subpartida,modelo,precision_prom,recall_prom,f1_prom
0,806100000-UVAS FRESCAS,IQR,0.564706,1.000000,0.721770
1,806100000-UVAS FRESCAS,IsolationForest,0.501963,1.000000,0.668350
2,806100000-UVAS FRESCAS,LOF,0.000000,0.000000,0.000000
4,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",IsolationForest,0.498631,1.000000,0.665368
3,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",IQR,0.825643,0.501832,0.624182
5,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",LOF,0.335653,0.501832,0.402198


In [6]:
# =============================================================================
# 6. MODELO GANADOR POR SUBPARTIDA
# =============================================================================

ganadores = (
    tabla_resumen
    .sort_values(["subpartida", "f1_prom"], ascending=[True, False])
    .groupby("subpartida")
    .head(1)
    .reset_index(drop=True)
)

ganadores

,subpartida,modelo,precision_prom,recall_prom,f1_prom
0,806100000-UVAS FRESCAS,IQR,0.564706,1.0,0.721770
1,"810400000-ARANDANOS ROJOS, MIRTILOS Y DEMAS FR...",IsolationForest,0.498631,1.0,0.665368


In [7]:
# =============================================================================
# 7. ALERTAS REALES SIMPLES CON IQR POR SUBPARTIDA
# =============================================================================

alertas = []

for subpartida, df_sub in df_demo.groupby("NUM_SPN_R"):
    q1 = df_sub["MTO_VALOR_UNTARIO_V"].quantile(0.25)
    q3 = df_sub["MTO_VALOR_UNTARIO_V"].quantile(0.75)
    iqr = q3 - q1
    li = q1 - 1.5 * iqr
    ls = q3 + 1.5 * iqr

    tmp = df_sub.copy()
    tmp["LIMITE_INFERIOR"] = li
    tmp["LIMITE_SUPERIOR"] = ls
    tmp["ES_ALERTA"] = ((tmp["MTO_VALOR_UNTARIO_V"] < li) | (tmp["MTO_VALOR_UNTARIO_V"] > ls)).astype(int)
    alertas.append(tmp[tmp["ES_ALERTA"] == 1])

alertas_reales = pd.concat(alertas, ignore_index=True) if alertas else pd.DataFrame()

print(f"Alertas reales detectadas: {len(alertas_reales):,}")

alertas_reales[[
    "ID_REGISTRO",
    "NUM_SPN_R",
    "ANIO_C",
    "MTO_VALOR_UNTARIO_V",
    "FOB_DOLAR",
    "PESO_NETO",
    "SECTOR",
    "TIPO_PRODUCTO",
    "ADUANA",
    "LIMITE_INFERIOR",
    "LIMITE_SUPERIOR",
]].head(20)

Alertas reales detectadas: 1,133


,ID_REGISTRO,NUM_SPN_R,ANIO_C,MTO_VALOR_UNTARIO_V,FOB_DOLAR,PESO_NETO,SECTOR,TIPO_PRODUCTO,ADUANA,LIMITE_INFERIOR,LIMITE_SUPERIOR
0,40248,806100000-UVAS FRESCAS,2024,222.870000,445.74,2.0,500,02 - PRODUCTOS NO TRADICIONALES,AEREA Y POSTAL EX - IAAC,1.289299,4.689835
1,95406,806100000-UVAS FRESCAS,2024,5.308627,88356.78,16644.0,500,02 - PRODUCTOS NO TRADICIONALES,MARITIMA DEL CALLAO,1.289299,4.689835
2,151914,806100000-UVAS FRESCAS,2024,1.098103,18276.82,16644.0,500,02 - PRODUCTOS NO TRADICIONALES,MARITIMA DEL CALLAO,1.289299,4.689835
3,154638,806100000-UVAS FRESCAS,2024,4.794521,79800.00,16644.0,500,02 - PRODUCTOS NO TRADICIONALES,PISCO,1.289299,4.689835
4,154664,806100000-UVAS FRESCAS,2024,4.931507,82080.00,16644.0,500,02 - PRODUCTOS NO TRADICIONALES,PISCO,1.289299,4.689835
5,155473,806100000-UVAS FRESCAS,2024,0.918000,7344.00,8000.0,500,02 - PRODUCTOS NO TRADICIONALES,DESAGUADERO,1.289299,4.689835
6,155474,806100000-UVAS FRESCAS,2024,0.980000,12544.00,12800.0,500,02 - PRODUCTOS NO TRADICIONALES,DESAGUADERO,1.289299,4.689835
7,155475,806100000-UVAS FRESCAS,2024,0.980000,15680.00,16000.0,500,02 - PRODUCTOS NO TRADICIONALES,DESAGUADERO,1.289299,4.689835
8,155476,806100000-UVAS FRESCAS,2024,0.963125,5085.30,5280.0,500,02 - PRODUCTOS NO TRADICIONALES,DESAGUADERO,1.289299,4.689835
9,155477,806100000-UVAS FRESCAS,2024,0.980000,5488.00,5600.0,500,02 - PRODUCTOS NO TRADICIONALES,DESAGUADERO,1.289299,4.689835


In [8]:
# =============================================================================
# 8. GRAFICO SIMPLE PARA LA PRESENTACION
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 5))

for subpartida, df_sub in df_demo.groupby("NUM_SPN_R"):
    ax.scatter(
        [subpartida] * len(df_sub),
        df_sub["MTO_VALOR_UNTARIO_V"],
        alpha=0.35,
        s=15,
    )

if not alertas_reales.empty:
    ax.scatter(
        alertas_reales["NUM_SPN_R"],
        alertas_reales["MTO_VALOR_UNTARIO_V"],
        s=45,
        marker="x",
        label="Alerta IQR",
    )

ax.set_yscale("log")
ax.set_title("Valor unitario por subpartida: cada subpartida es un universo")
ax.set_xlabel("Subpartida")
ax.set_ylabel("Valor unitario FOB / Peso neto")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.


Error in callback <function _draw_all_if_interactive at 0x0000025670B419E0> (for post_execute), with arguments args (),kwargs {}:


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.


ValueError: Failed to find font DejaVu Sans:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0, and fallback to the default font was disabled

<Figure size 1200x500 with 1 Axes>

## Lectura para exponer

1. Primero se carga la data del procedimiento `EDA_BASE`.
2. La demostracion se concentra solo en dos subpartidas: `806100000` y `810400000`.
3. Cada subpartida se trabaja como un universo independiente.
4. Los folds se crean dentro de cada subpartida, sin mezclar productos.
5. Se comparan tres metodos: IQR, Isolation Forest y LOF.
6. El cuadro `tabla_resumen` muestra el resultado por subpartida y modelo.
7. Las alertas reales son casos priorizados para revision tecnica, no una conclusion de fraude.
